Setup & Configuration

In [ ]:
# !pip install -q mne mne-bids mne-icalabel specparam openneuro-py statsmodels seaborn

import os, re, glob, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import statsmodels.formula.api as smf
from scipy import stats
from mne.preprocessing import ICA
from mne_icalabel import label_components
from specparam import SpectralModel

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11})

# Paths (adjust BASE_DIR if running locally)
BASE_DIR = '/content'
RAW_DIR  = f'{BASE_DIR}/mtbi_rest'
DATA_DIR = f'{BASE_DIR}/data'
FIG_DIR  = f'{BASE_DIR}/figures'
for d in [RAW_DIR, DATA_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

ROIS = {
    'Frontal':  ['F3', 'Fz', 'F4', 'FC1', 'FC2'],
    'Central':  ['C3', 'Cz', 'C4', 'CP1', 'CP2'],
    'Parietal': ['P3', 'Pz', 'P4', 'PO3', 'PO4']
}
POSTERIOR_CHS = ['O1', 'Oz', 'O2', 'PO3', 'PO4', 'P3', 'Pz', 'P4']
GROUP_MAP = {1: 'Control', 0: 'Subacute mTBI', 2: 'Chronic TBI'}

Fetch BIDS Metadata (ds003522)

In [ ]:
# Download lightweight metadata and JSON headers
!openneuro-py download \
    --dataset=ds003522 \
    --target-dir={RAW_DIR} \
    --include=participants.tsv \
    --include=dataset_description.json \
    --include=sub-*/**/eeg/*_eeg.json

df_meta = pd.read_csv(f'{RAW_DIR}/participants.tsv', sep='\t')
df_meta['Group_Label'] = df_meta['Group'].map(GROUP_MAP)

eeg_jsons = glob.glob(f'{RAW_DIR}/sub-*/**/eeg/*_eeg.json', recursive=True)
records = []
for f in eeg_jsons:
    parts = os.path.basename(f).split('_')
    records.append({
        'participant_id': next((p for p in parts if p.startswith('sub-')), None),
        'session': next((p for p in parts if p.startswith('ses-')), 'none'),
        'task': next((p for p in parts if p.startswith('task-')), 'none')
    })

df_files = pd.DataFrame(records)
print(f"Participants: {len(df_meta)} | Total EEG recordings: {len(df_files)}")
print(pd.crosstab(df_files.merge(df_meta[['participant_id', 'Group_Label']], on='participant_id')['Group_Label'],
                  df_files['session']))

Section 1: Full ses-01 Aperiodic ($1/f$) Extraction & LMMs

In [ ]:
def extract_rest_rois(sub_id, ses='01', root=RAW_DIR):
    eeg_dir = f"{root}/{sub_id}/ses-{ses}/eeg"
    set_files = glob.glob(f"{eeg_dir}/*_eeg.set")
    if not set_files:
        os.system(f"openneuro-py download --dataset=ds003522 --target-dir={root} --include={sub_id}/ses-{ses}/eeg/* >/dev/null 2>&1")
        set_files = glob.glob(f"{eeg_dir}/*_eeg.set")
    if not set_files:
        raise FileNotFoundError(f"No .set file for {sub_id} ses-{ses}")

    raw = mne.io.read_raw_eeglab(set_files[0], preload=True)
    drop_chs = [ch for ch in raw.ch_names if any(k in ch.upper() for k in ['EOG', 'ECG', 'EKG', 'M1', 'M2', 'CB1', 'CB2'])]
    if drop_chs:
        raw.drop_channels(drop_chs)
    raw.set_montage('standard_1020', on_missing='ignore')
    raw.resample(250)
    raw.filter(1.0, 100.0, fir_design='firwin')
    raw.notch_filter(60.0)

    events, event_id = mne.events_from_annotations(raw)
    eo_keys = [k for k in event_id if re.fullmatch(r'(Stimulus/)?S\s*1', k.strip())]
    ec_keys = [k for k in event_id if re.fullmatch(r'(Stimulus/)?S\s*3', k.strip())]
    if not eo_keys or not ec_keys:
        raise ValueError(f"Missing rest triggers in {sub_id}")

    rest_id = {'Eyes_Open': event_id[eo_keys[0]], 'Eyes_Closed': event_id[ec_keys[0]]}
    epochs = mne.Epochs(raw, events, event_id=rest_id, tmin=0.0, tmax=2.0,
                        baseline=None, preload=True, reject_by_annotation=True)

    # Robust MAD-based bad channel interpolation
    data = epochs.get_data()
    ch_stds = np.median(np.std(data, axis=2), axis=0)
    med, mad = np.median(ch_stds), np.median(np.abs(ch_stds - np.median(ch_stds)))
    bad_idx = np.where((ch_stds > med + 5 * 1.4826 * mad) | (ch_stds < 1e-7))[0]
    if len(bad_idx) > 0:
        epochs.info['bads'] = [epochs.ch_names[i] for i in bad_idx]
        epochs.interpolate_bads(reset_bads=True)

    # Average reference + Extended Infomax ICA + ICLabel
    epochs.set_eeg_reference('average', projection=False)
    ica = ICA(n_components=20, method='infomax', fit_params=dict(extended=True), random_state=42, max_iter=300)
    ica.fit(epochs)
    ic_labels = label_components(epochs, ica, method='iclabel')
    ica.exclude = [
        i for i, (lbl, prob) in enumerate(zip(ic_labels['labels'], ic_labels['y_pred_proba']))
        if lbl in ['eye blink', 'muscle artifact', 'heart beat', 'line noise', 'channel noise'] and prob >= 0.7
    ]
    epochs_clean = ica.apply(epochs.copy())

    rows = []
    for cond in ['Eyes_Closed', 'Eyes_Open']:
        ep_cond = epochs_clean[cond]
        psd = ep_cond.compute_psd(method='welch', fmin=1.0, fmax=45.0, n_fft=500)
        freqs, psds = psd.freqs, psd.get_data().mean(axis=0)

        for roi_name, ch_list in ROIS.items():
            present = [ch for ch in ch_list if ch in ep_cond.ch_names]
            if not present:
                continue
            exps, offs, r2s = [], [], []
            for ch in present:
                sm = SpectralModel(peak_width_limits=[1.0, 8.0], max_n_peaks=6, min_peak_height=0.1, verbose=False)
                sm.fit(freqs, psds[ep_cond.ch_names.index(ch)], [2, 40])
                ap = sm.get_params('aperiodic')
                exps.append(ap[1]); offs.append(ap[0]); r2s.append(sm.get_metrics('gof', 'rsquared'))

            rows.append({
                'participant_id': sub_id, 'Session': f'ses-{ses}', 'Condition': cond,
                'Region': roi_name, 'Exponent_2_40': np.nanmedian(exps),
                'Offset_2_40': np.nanmedian(offs), 'Median_R2': np.nanmedian(r2s),
                'N_Epochs': len(ep_cond), 'N_ICs_Excluded': len(ica.exclude)
            })

    del raw, epochs, epochs_clean, ica
    gc.collect()
    return rows

# Resumable full ses-01 sweep
csv_ap = f'{DATA_DIR}/ds003522_rest_pilot.csv'
done_subs = set(pd.read_csv(csv_ap)['participant_id'].unique()) if os.path.exists(csv_ap) else set()
remaining = [s for s in df_meta['participant_id'] if s not in done_subs]

for i, sub_id in enumerate(remaining):
    try:
        df_sub = pd.DataFrame(extract_rest_rois(sub_id, ses='01'))
        df_sub.to_csv(csv_ap, mode='a', header=not os.path.exists(csv_ap), index=False)
    except Exception as e:
        print(f"SKIPPED {sub_id}: {e}")

# LMMs: 1/f Exponent & Offset (QC R² >= 0.90)
df_ap = pd.read_csv(csv_ap).merge(df_meta[['participant_id', 'Group_Label', 'age', 'sex']], on='participant_id', how='left')
df_ap_qc = df_ap[df_ap['Median_R2'] >= 0.90].copy()

for metric in ['Exponent_2_40', 'Offset_2_40']:
    for cond in ['Eyes_Closed', 'Eyes_Open']:
        sub_df = df_ap_qc[df_ap_qc['Condition'] == cond]
        lmm = smf.mixedlm(f"{metric} ~ C(Group_Label, Treatment('Control')) + C(Region) + age + sex",
                          data=sub_df, groups=sub_df['participant_id']).fit()
        print(f"\n=== {cond} LMM: {metric} ===")
        print(lmm.summary().tables[1])

Section 2: Cross-Sectional Posterior Alpha (ses-01, All 3 Groups)

In [ ]:
csv_alpha_s1 = f'{DATA_DIR}/ds003522_ses01_alpha_allgroups.csv'

if not os.path.exists(csv_alpha_s1):
    rows = []
    for fpath in sorted(glob.glob(f"{RAW_DIR}/sub-*/ses-01/eeg/*_eeg.set")):
        sub_id = os.path.basename(fpath).split('_')[0]
        try:
            raw = mne.io.read_raw_eeglab(fpath, preload=True)
            present = [ch for ch in POSTERIOR_CHS if ch in raw.ch_names]
            if not present:
                continue
            raw.pick_channels(present)
            raw.resample(250)
            raw.filter(1.0, 45.0, fir_design='firwin')

            events, event_id = mne.events_from_annotations(raw)
            ec_keys = [k for k in event_id if re.fullmatch(r'(Stimulus/)?S\s*3', k.strip())]
            if not ec_keys:
                continue

            epochs = mne.Epochs(raw, events, event_id={'EC': event_id[ec_keys[0]]},
                                tmin=0.0, tmax=2.0, baseline=None, preload=True, reject_by_annotation=True)
            psd = epochs.compute_psd(method='welch', fmin=1.0, fmax=45.0, n_fft=500)
            sm = SpectralModel(peak_width_limits=[1.0, 8.0], max_n_peaks=6, min_peak_height=0.1, verbose=False)
            sm.fit(psd.freqs, psd.get_data().mean(axis=(0, 1)), [2, 40])
            peaks = sm.get_params('peak')

            alpha_cf, alpha_pw = np.nan, np.nan
            if peaks is not None and len(peaks) > 0:
                peaks = np.atleast_2d(peaks)
                apks = peaks[(peaks[:, 0] >= 7.0) & (peaks[:, 0] <= 13.0)]
                if len(apks) > 0:
                    best = apks[np.argmax(apks[:, 1])]
                    alpha_cf, alpha_pw = best[0], best[1]

            rows.append({'participant_id': sub_id, 'Alpha_CF': alpha_cf, 'Alpha_PW': alpha_pw,
                         'R2': sm.get_metrics('gof', 'rsquared')})
            del raw, epochs; gc.collect()
        except Exception:
            continue

    df_alpha = pd.DataFrame(rows).merge(df_meta[['participant_id', 'Group', 'Group_Label', 'age', 'sex']], on='participant_id', how='left')
    df_alpha[(df_alpha['R2'] >= 0.90) & (df_alpha['Alpha_CF'].notna())].to_csv(csv_alpha_s1, index=False)

df_iaf_s1 = pd.read_csv(csv_alpha_s1)
df_iaf_s1['Group_Label'] = df_iaf_s1['Group'].map(GROUP_MAP)
df_iaf_s1['Any_TBI'] = (df_iaf_s1['Group_Label'] != 'Control').astype(int)

print("=== 3-Group OLS: Posterior Alpha CF ===")
print(smf.ols("Alpha_CF ~ C(Group_Label, Treatment('Control')) + age + sex", data=df_iaf_s1).fit().summary().tables[1])
print("\n=== Pooled TBI vs. Control OLS: Posterior Alpha CF ===")
print(smf.ols("Alpha_CF ~ Any_TBI + age + sex", data=df_iaf_s1).fit().summary().tables[1])

Section 3: High-Res Longitudinal Alpha (ses-01 → ses-03) & Attrition Audit

In [ ]:
csv_long = f'{DATA_DIR}/ds003522_longitudinal_alpha_hires.csv'
long_subs = df_meta[df_meta['Group'].isin([0, 1])]['participant_id'].tolist()
target_pairs = df_files[df_files['participant_id'].isin(long_subs)][['participant_id', 'session']].drop_duplicates()

done_pairs = set(zip(pd.read_csv(csv_long)['participant_id'], pd.read_csv(csv_long)['Session'])) if os.path.exists(csv_long) else set()

for row in target_pairs.itertuples(index=False):
    sub_id, ses = row.participant_id, row.session
    if (sub_id, ses) in done_pairs:
        continue

    eeg_dir = f"{RAW_DIR}/{sub_id}/{ses}/eeg"
    set_files = glob.glob(f"{eeg_dir}/*_eeg.set")
    if not set_files:
        os.system(f"openneuro-py download --dataset=ds003522 --target-dir={RAW_DIR} --include={sub_id}/{ses}/eeg/* >/dev/null 2>&1")
        set_files = glob.glob(f"{eeg_dir}/*_eeg.set")
    if not set_files:
        continue

    try:
        raw = mne.io.read_raw_eeglab(set_files[0], preload=True)
        present = [ch for ch in POSTERIOR_CHS if ch in raw.ch_names]
        if not present:
            continue
        raw.pick_channels(present)
        raw.resample(250)
        raw.filter(1.0, 45.0, fir_design='firwin')
        raw.set_eeg_reference('average', projection=False)

        events, event_id = mne.events_from_annotations(raw)
        ec_keys = [k for k in event_id if re.fullmatch(r'(Stimulus/)?S\s*3', k.strip())]
        if not ec_keys:
            continue

        epochs = mne.Epochs(raw, events, event_id={'EC': event_id[ec_keys[0]]},
                            tmin=0.0, tmax=2.0, baseline=None, preload=True, reject_by_annotation=True)
        psd = epochs.compute_psd(method='welch', fmin=1.0, fmax=45.0, n_fft=2000, n_per_seg=500)
        sm = SpectralModel(peak_width_limits=[1.0, 8.0], max_n_peaks=6, min_peak_height=0.1, verbose=False)
        sm.fit(psd.freqs, psd.get_data().mean(axis=(0, 1)), [2, 40])
        peaks = sm.get_params('peak')

        alpha_cf, alpha_pw = np.nan, np.nan
        if peaks is not None and len(peaks) > 0:
            peaks = np.atleast_2d(peaks)
            apks = peaks[(peaks[:, 0] >= 7.0) & (peaks[:, 0] <= 13.0)]
            if len(apks) > 0:
                best = apks[np.argmax(apks[:, 1])]
                alpha_cf, alpha_pw = best[0], best[1]

        pd.DataFrame([{
            'participant_id': sub_id, 'Session': ses, 'Alpha_CF': alpha_cf,
            'Alpha_PW': alpha_pw, 'Exponent': sm.get_params('aperiodic')[1],
            'R2': sm.get_metrics('gof', 'rsquared')
        }]).to_csv(csv_long, mode='a', header=not os.path.exists(csv_long), index=False)
        del raw, epochs; gc.collect()
    except Exception:
        continue

# Longitudinal LMM
df_long = pd.read_csv(csv_long).merge(df_meta[['participant_id', 'Group_Label', 'age', 'sex']], on='participant_id', how='left')
df_long_qc = df_long[(df_long['R2'] >= 0.90) & (df_long['Alpha_CF'].notna())].copy()

lmm_iaf = smf.mixedlm("Alpha_CF ~ C(Group_Label, Treatment('Control')) * C(Session) + age + sex",
                      data=df_long_qc, groups=df_long_qc['participant_id']).fit()
print("=== Longitudinal LMM: Posterior Alpha CF ===")
print(lmm_iaf.summary().tables[1])

# Paired completer & attrition audit
piv = df_long_qc.pivot(index=['participant_id', 'Group_Label'], columns='Session', values='Alpha_CF').reset_index()
print("\n=== Paired Completer Audit (ses-01 -> ses-02) ===")
for grp in ['Subacute mTBI', 'Control']:
    sub = piv[(piv['Group_Label'] == grp) & piv['ses-01'].notna() & piv['ses-02'].notna()]
    t_stat, p_val = stats.ttest_rel(sub['ses-02'], sub['ses-01'])
    print(f"{grp:14s} (n={len(sub):2d}): {sub['ses-01'].mean():.3f} -> {sub['ses-02'].mean():.3f} Hz | Δ={(sub['ses-02']-sub['ses-01']).mean():+.3f} (t={t_stat:+.2f}, p={p_val:.4f})")

mtbi = piv[piv['Group_Label'] == 'Subacute mTBI']
print(f"mTBI ses-01 IAF: Returners (n={mtbi['ses-02'].notna().sum()}) = {mtbi[mtbi['ses-02'].notna()]['ses-01'].mean():.3f} Hz | Dropouts (n={mtbi['ses-02'].isna().sum()}) = {mtbi[mtbi['ses-02'].isna()]['ses-01'].mean():.3f} Hz")

Section 4:Figures

In [ ]:
palette = {'Control': '#4C72B0', 'Subacute mTBI': '#DD8452', 'Chronic TBI': '#C44E52'}
order_3 = ['Control', 'Subacute mTBI', 'Chronic TBI']

# Figure 1: Double Dissociation
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
eo_subj = df_ap_qc[df_ap_qc['Condition'] == 'Eyes_Open'].groupby(
    ['participant_id', 'Group_Label', 'age'], as_index=False)['Exponent_2_40'].mean()

sns.scatterplot(data=eo_subj, x='age', y='Exponent_2_40', hue='Group_Label',
                hue_order=order_3, palette=palette, s=55, alpha=0.8, ax=axes[0])
sns.regplot(data=eo_subj, x='age', y='Exponent_2_40', scatter=False,
            color='#222222', line_kws={'lw': 2, 'ls': '--'}, ax=axes[0])
axes[0].set_title("A. Aperiodic 1/f Exponent Tracks Aging, Not TBI")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Mean ROI 1/f Exponent (2–40 Hz, Eyes Open)")
axes[0].legend(title="", frameon=True)

sns.boxplot(data=df_iaf_s1, x='Group_Label', y='Alpha_CF', order=order_3,
            palette=palette, width=0.5, showfliers=False, ax=axes[1])
sns.stripplot(data=df_iaf_s1, x='Group_Label', y='Alpha_CF', order=order_3,
              color='#222222', alpha=0.6, size=5, ax=axes[1])
axes[1].set_title("B. Posterior Alpha Frequency Tracks TBI Slowing")
axes[1].set_xlabel("")
axes[1].set_ylabel("Posterior Alpha Center Frequency (Hz, Eyes Closed)")

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_double_dissociation.png', dpi=300, bbox_inches='tight')
plt.show()

# Figure 2: Longitudinal Trajectory & Informative Attrition
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.pointplot(data=df_long_qc, x='Session', y='Alpha_CF', hue='Group_Label',
              order=['ses-01', 'ses-02', 'ses-03'], hue_order=['Control', 'Subacute mTBI'],
              palette=palette, dodge=0.15, capsize=0.1, errorbar='se', ax=axes[0])
axes[0].set_title("A. Longitudinal Trajectory (All Available Sessions)")
axes[0].set_xlabel("Session (ses-01: Acute | ses-02: ~2 mo | ses-03: ~4 mo)")
axes[0].set_ylabel("Posterior Alpha Center Frequency (Hz)")
axes[0].legend(title="", frameon=True)

mtbi_piv = piv[piv['Group_Label'] == 'Subacute mTBI'].copy()
mtbi_piv['Retention'] = np.where(mtbi_piv['ses-02'].notna(), 'Returned (ses-02)', 'Dropped Out')
sns.boxplot(data=mtbi_piv, x='Retention', y='ses-01', order=['Returned (ses-02)', 'Dropped Out'],
            palette={'Returned (ses-02)': '#DD8452', 'Dropped Out': '#8C8C8C'},
            width=0.45, showfliers=False, ax=axes[1])
sns.stripplot(data=mtbi_piv, x='Retention', y='ses-01', order=['Returned (ses-02)', 'Dropped Out'],
              color='#222222', alpha=0.65, size=5.5, ax=axes[1])
axes[1].set_title("B. Informative Attrition in Subacute mTBI (ses-01)")
axes[1].set_xlabel("")
axes[1].set_ylabel("Baseline (ses-01) Alpha Center Frequency (Hz)")

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_longitudinal_and_attrition.png', dpi=300, bbox_inches='tight')
plt.show()